In [ ]:
import os
from huggingface_hub import login

os.environ["WANDB_DISABLED"] = "true"

login("YOUR_HF_ACCESS_TOKEN")

In [ ]:
from transformers import AutoTokenizer

model_name = "google-bert/bert-base-cased"              # BERT
# model_name = "FacebookAI/roberta-base"                  # RoBERTa

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
tokenizer.is_fast

In [ ]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
import pandas as pd

# Chọn uncomment 1 trong các file sau để chọn training set tương ứng cho từng thực nghiệm

df_train = pd.read_csv(r"/root/train.csv")                              # w/o augmentation
# df_train = pd.read_csv(r"/root/train_eda.csv")                          # EDA augmentation
# df_train = pd.read_csv(r"/root/train_aeda.csv")                         # AEDA augmentation
# df_train = pd.read_csv(r"/root/train_ease.csv")                         # EASE augmentation
# df_train = pd.read_csv(r"/root/train-llm-aug-gpt-5-2025-08-07.csv")     # LLM augmentation (gpt-5-2025-08-07)

df_val = pd.read_csv("/root/test.csv")
df_test = pd.read_csv("/root/val.csv")

df_train = df_train[["text", "priority"]]
df_val = df_val[["text", "priority"]]
df_test = df_test[["text", "priority"]]

# ===== UNDER_SAMPLING WITH REAL TRAINING DATA =====
# def process_training_set_with_balance_class_distribution(dataframe, target_column_name="priority"):
#     min_sample = dataframe[target_column_name].value_counts().min()
    
#     df_filtered = dataframe.groupby(target_column_name, group_keys=False).apply(
#         lambda x: x.sample(n=min_sample, random_state=42)
#     ).reset_index(drop=True)
    
#     return df_filtered

# df_train = process_training_set_with_balance_class_distribution(dataframe=df_train)

# ===== OVER_SAMPLING WITH REAL TRAINING DATA =====
# def process_training_set_with_oversampling(dataframe, target_column_name="priority"):
#     max_sample = dataframe[target_column_name].value_counts().max()
    
#     df_oversampled = dataframe.groupby(target_column_name, group_keys=False).apply(
#         lambda x: x.sample(n=max_sample, replace=True, random_state=42)
#     ).reset_index(drop=True)
    
#     return df_oversampled

# df_train = process_training_set_with_oversampling(dataframe=df_train)

df_train.columns = ["text", "labels"]
df_val.columns = ["text", "labels"]
df_test.columns =["text", "labels"]

In [ ]:
df_train["labels"].value_counts()

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {
    0: "Trivial",
    1: "Minor",
    2: "Major",
    3: "Critical",
    4: "Blocker"
}

label2id = {
    'Trivial': 0,
    'Minor': 1,
    'Major': 2,
    'Critical': 3,
    'Blocker': 4
}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
def preprocess_function(examples):
    if hasattr(model.config, 'max_position_embeddings'):
        max_len = min(model.config.max_position_embeddings - 2, 512)
    else:
        max_len = 512
    
    tokenized = tokenizer(
        examples["text"], 
        truncation=True, 
        padding=True,
        max_length=max_len,
        return_tensors=None
    )
    tokenized["labels"] = [label2id[label] for label in examples["labels"]]
    return tokenized

In [ ]:
from datasets import Dataset

ds_train = Dataset.from_pandas(df_train)
ds_val = Dataset.from_pandas(df_val)
ds_test = Dataset.from_pandas(df_test)

tokenized_train = ds_train.map(preprocess_function, batched=True)
tokenized_val = ds_val.map(preprocess_function, batched=True)
tokenized_test = ds_test.map(preprocess_function, batched=True)

In [ ]:
!pip install evaluate -q

In [ ]:
import evaluate

f1_metric = evaluate.load("f1")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="my_awesome_model",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,              # Tăng epochs để thấy rõ progress
    weight_decay=0.01,
    
    # Evaluation settings
    evaluation_strategy="steps",
    eval_steps=100,
    
    # Logging settings 
    logging_strategy="steps",
    logging_steps=100,                # Log training loss mỗi 100 steps
    
    # Save settings
    save_strategy="steps", 
    save_steps=100,
    save_total_limit=3,              # Chỉ giữ 3 checkpoints gần nhất
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",  # Chọn best model dựa trên accuracy
    greater_is_better=True,

    warmup_steps=300, #300
    lr_scheduler_type="cosine",
    
    push_to_hub=False,
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
import numpy as np

In [ ]:
trainer.train()

In [ ]:
test_results = trainer.predict(tokenized_test)
predictions = np.argmax(test_results.predictions, axis=1)
test_labels = test_results.label_ids

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

print("Accuracy:", round(accuracy_score(test_labels, predictions)*100, 2))
print("Macro-Precision:", round(precision_score(test_labels, predictions, average='macro')*100, 2))
print("Macro-Recall:", round(recall_score(test_labels, predictions, average='macro')*100, 2))
print("Macro-F1 score:", round(f1_score(test_labels, predictions, average='macro')*100, 2))
print("="*50)
print("Micro-Precision:", round(precision_score(test_labels, predictions, average='micro')*100, 2))
print("Micro-Recall:", round(recall_score(test_labels, predictions, average='micro')*100, 2))
print("Micro-F1 score:", round(f1_score(test_labels, predictions, average='micro')*100, 2))